# ASSERT reproduction on SafeText with Qwen2.5-7B-Instruct-AWQ

Independent reproduction of *ASSERT* (Mei, Levy, Wang; Findings of EMNLP 2023). **Run all cells top to bottom on a Colab T4 GPU runtime.**
A clean run takes about 1 hour (~18,000 model calls).

**What is faithful to the paper and what is a Qwen adaptation** (full table in the README):

| Paper protocol (kept) | Qwen adaptation (changed) |
|---|---|
| SafeText, 4-shot Yes/No + rationale, greedy decoding, two-proportion z-test | Model: Qwen2.5-7B-Instruct, 4-bit AWQ (paper: GPT-3/ChatGPT/GPT-4/Alpaca/Vicuna) |
| SAA: 5 paraphrases; AKI: hints, zero- and 4-shot; TB: 2-step bootstrapping | `max_tokens` 256; system message first; TB at 3x3 with extra system message + strict cleaner |
| TB on the authors' released samples | Added: cluster bootstrap over source scenarios |

**Sections:** 1 Configure · 2 Environment · 3 Code and data · 4 Model server · 5 Run the experiments · 6 Verify the run · 7 Tables · 8 Save and export · 9 Optional diagnostics.

Safe to re-run after a disconnect: finished stages are skipped and the cache is reused.

## 1. Configure (the only cell you edit)

In [ ]:
# ======================= EDIT ONLY THIS CELL =======================
RUN_ID = "run1-clean"            # new folder: MyDrive/assert/runs/<RUN_ID>
MODE = "clean"                   # "clean"  = start with an EMPTY cache  -> genuine reproduction (~1 h)
                                 # "replay" = start from a saved cache   -> minutes, replays old answers
REPLAY_CACHE = "/content/drive/MyDrive/assert/archive/outputs_final/llm_cache.jsonl"   # used only if MODE == "replay"

GITHUB_USER, REPO_NAME = "rohailforty2", "assert-qwen-repro"
REPO_REF = "main"                # branch, tag or commit of YOUR repo to run
AUTHORS_REPO_REF = "1cafa82"     # pinned commit of github.com/alexmeigz/ASSERT (no license: cloned, never committed)

MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"
MODEL_REVISION = "b25037543e9394b818fdfca67ab2a00ecc7dd641"   # Hugging Face commit of the weights (checked Oct 2026)
TB_N, TB_M, WORKERS = 3, 3, 32   # TB: new contexts per scenario, new actions per context; parallel requests
# ====================================================================
assert MODE in ("clean", "replay"), "MODE must be 'clean' or 'replay'"
print("RUN_ID:", RUN_ID, "| MODE:", MODE)

## 2. Environment

In [ ]:
import subprocess, sys

def run(*cmd, cwd="/content/repo"):
    """Run a command and print its output live in the cell. Stops the notebook if it fails."""
    p = subprocess.Popen(list(cmd), cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    code = p.wait()
    assert code == 0, f"command failed (exit {code}): {' '.join(cmd)}"

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print("GPU:", gpu.stdout.strip() or "NONE")
assert gpu.returncode == 0 and gpu.stdout.strip(), "No GPU. Runtime > Change runtime type > T4 GPU, then run again."

from google.colab import drive
drive.mount("/content/drive")
import os
RUN = f"/content/drive/MyDrive/assert/runs/{RUN_ID}"
os.makedirs(RUN, exist_ok=True)
print("run folder:", RUN)
print("already there:", sorted(os.listdir(RUN)) or "(empty: fresh run)")

## 3. Code and data

Your code is cloned from GitHub (a private repo needs a read-only token stored in Colab **Secrets** as `GITHUB_TOKEN`; a public repo needs nothing).
The authors' repo is cloned at a pinned commit and **never committed** (it has no license).

In [ ]:
import os, subprocess

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None                      # public repo or no secret set

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        err = r.stderr.replace(token, "***") if token else r.stderr
        raise RuntimeError(err[-800:])
    return r.stdout.strip()

for d in ("/content/repo", "/content/ASSERT"):
    sh(f"rm -rf {d}")
auth = f"{token}@" if token else ""
sh(f"git clone -q https://{auth}github.com/{GITHUB_USER}/{REPO_NAME}.git /content/repo")
sh(f"git -C /content/repo remote set-url origin https://github.com/{GITHUB_USER}/{REPO_NAME}.git")  # drop the token
sh(f"git -C /content/repo checkout -q {REPO_REF}")
sh("git clone -q https://github.com/alexmeigz/ASSERT /content/ASSERT")
sh(f"git -C /content/ASSERT checkout -q {AUTHORS_REPO_REF}")
print("repo   :", sh("git -C /content/repo log -1 --format='%h %s'"))
print("authors:", sh("git -C /content/ASSERT log -1 --format='%h %s'"))

In [ ]:
%pip install -q openai numpy vllm huggingface_hub

In [ ]:
# offline self-test with a mock LLM: must print "end-to-end smoke test passed" (the cell fails otherwise)
run("python", "tests/smoke_test.py", "/content/ASSERT")

## 4. Model server (vLLM)

Weights are pinned to `MODEL_REVISION`. Start-up takes a few minutes.

In [ ]:
import subprocess, time, requests
log = open("/content/vllm.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", MODEL, "--revision", MODEL_REVISION, "--port", "8000", "--dtype", "half",
     "--max-model-len", "4096", "--enable-prefix-caching"],
    stdout=log, stderr=log)
for _ in range(180):
    try:
        if requests.get("http://localhost:8000/health", timeout=3).status_code == 200:
            print("server up"); break
    except Exception:
        pass
    time.sleep(5)
else:
    print(open("/content/vllm.log").read()[-3000:])
    raise RuntimeError("vLLM did not start; see the log above")

## 5. Run the experiments

`MODE = "clean"` starts with an empty cache. `MODE = "replay"` copies a saved cache first.
The pipeline refuses a second launch on the same folder (lock file) and skips finished stages.

In [ ]:
import shutil
cache = f"{RUN}/llm_cache.jsonl"
if MODE == "replay" and not os.path.exists(cache):
    assert os.path.exists(REPLAY_CACHE), f"replay cache not found: {REPLAY_CACHE}"
    shutil.copy(REPLAY_CACHE, cache)
n = sum(1 for _ in open(cache, "rb")) if os.path.exists(cache) else 0
print(f"cache lines at start: {n}  ({'resuming/replay' if n else 'empty: every answer comes from the model'})")

In [ ]:
import subprocess, os
def pipeline(*extra):
    cmd = ["python", "run_pipeline.py", "--repo", "/content/ASSERT", "--out", RUN,
           "--model", MODEL, "--base-url", "http://localhost:8000/v1", "--api-key", "EMPTY",
           "--tb-n", str(TB_N), "--tb-m", str(TB_M), "--workers", str(WORKERS), *extra]
    run(*cmd)                                           # prints live, fails loudly

# (a) baseline + SAA + TB with our own generation + AKI.  Takes most of the time.
pipeline("--tb-source", "generated")

In [ ]:
# (b) TB on the authors' released scenarios (reuses everything finished above)
pipeline("--tb-source", "authors")
print("pipeline finished")

## 6. Verify the run (must say PASS before you trust any number)

In [ ]:
run("python", "scripts/check_run.py", "--out", RUN)   # raises if any check FAILs

## 7. Tables

In [ ]:
run("python", "scripts/make_report.py", "--out", RUN)

In [ ]:
run("python", "scripts/bootstrap.py", "--out", RUN, "--boot-b", "10000")

## 8. Save and export

Writes `manifest.json` (commit hashes, versions, GPU, file hashes) and builds a small zip with only the publishable summary files.
Unzip it into your repo as `results/<RUN_ID>/`, then commit. **Do not commit** the `.jsonl` files or the cache.

In [ ]:
run("python", "scripts/make_manifest.py", "--out", RUN, "--model", MODEL,
    "--revision", MODEL_REVISION, "--authors-repo", "/content/ASSERT")

In [ ]:
import shutil, os
from google.colab import files
stage = f"/content/results/{RUN_ID}"
shutil.rmtree("/content/results", ignore_errors=True)
os.makedirs(stage)
for f in ("report_all.md", "bootstrap_report.md", "manifest.json"):
    shutil.copy(f"{RUN}/{f}", stage)
zip_path = shutil.make_archive(f"/content/results_{RUN_ID}", "zip", "/content/results")
print("summary files:", sorted(os.listdir(stage)))
files.download(zip_path)

## 9. Optional diagnostics (prints generated unsafe text)

Inspect what TB generated. **Clear this cell's output before committing the notebook.**

In [ ]:
import json, random
load = lambda n: [json.loads(l) for l in open(f"{RUN}/{n}.jsonl", encoding="utf-8") if l.strip()]
g = load("tb_gen")
k = min(12, len(g))
print(len(g), f"generated TB scenarios; {k} random ones (context | action):\n")
for r in random.Random(1).sample(g, k):
    print(" -", r["prompt"], "|", r["advice"])